# FOMO — SJSU Headcount Training & Visualization

End-to-end notebook that:
1. Clones [fomo-edge-ai/fomo](https://github.com/fomo-edge-ai/fomo) and installs it
2. Downloads the **SJSU Headcount Scene-1** dataset from HuggingFace (`bdanko/sjsu-headcount-scene-1`)
3. Trains **FOMO-m** (the lightweight MobileNetV2 point-localization model) for 10 epochs
4. Visualizes the predictions on validation images — predicted person locations as coloured circles overlaid on the original image

## 1 — Environment Setup

In [ ]:
!pip install fomo-edge-ai==0.0.10

You must restart the session after running the above

In [ ]:

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


## 2 — Download Dataset

`bdanko/sjsu-headcount-scene-1` is a YOLO-format dataset (standard `data.yaml` + `train/images/` + `valid/images/` layout).
It is cloned via `git` the same way all other FOMO training datasets are fetched (marbles, RF5, etc.).

In [ ]:
import zipfile
from pathlib import Path
import yaml
from huggingface_hub import hf_hub_download

DATASET_ROOT = Path("sjsu-headcount-scene-1")
HF_REPO = "bdanko/sjsu-headcount-scene-1"
ZIP_NAME = "sjsu-headcount-scene-1.zip"

if not DATASET_ROOT.exists():
    print(f"Downloading dataset archive {ZIP_NAME} from {HF_REPO}...")
    zip_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=ZIP_NAME,
        repo_type="dataset"
    )
    print(f"Extracting to {DATASET_ROOT}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATASET_ROOT)
    print("Extraction complete.")
else:
    print("Dataset directory already exists — skipping download")

# Patch data.yaml: replace relative path '.' with the absolute cloned directory
data_yaml_path = DATASET_ROOT / "data.yaml"
data_cfg = yaml.safe_load(data_yaml_path.read_text())
data_cfg["path"] = str(DATASET_ROOT.resolve())
data_yaml_path.write_text(yaml.dump(data_cfg, default_flow_style=False))

print(f"\nDataset config:")
print(f"  path : {data_cfg['path']}")
print(f"  train: {data_cfg['train']}")
print(f"  val  : {data_cfg.get('val', data_cfg.get('valid', 'N/A'))}")
print(f"  nc   : {data_cfg['nc']} ({data_cfg['names']})")

# Quick sanity check — count images and labels
def count_split(split_dir):
    imgs = list(split_dir.rglob("*.jpg")) + list(split_dir.rglob("*.png"))
    lbl_dir = split_dir.parent / "labels"
    lbls = list(lbl_dir.rglob("*.txt")) if lbl_dir.exists() else []
    return len(imgs), len(lbls)

train_imgs, train_lbls = count_split(DATASET_ROOT / "train" / "images")
val_imgs, val_lbls     = count_split(DATASET_ROOT / "valid" / "images")
print(f"Train : {train_imgs} images, {train_lbls} label files")
print(f"Val   : {val_imgs} images, {val_lbls} label files")



## 3 — Train FOMO-m

FOMO-m uses a **MobileNetV2 backbone (α=0.5)** with a single-pixel detection head.
Input resolution is **192** → output grid **24x24** (8× downsample).

Training uses:
- Adam, lr=3e-4, ReduceLROnPlateau on val F1
- Weighted CrossEntropy (background=1×, foreground=100×)
- No augmentation (plain resize + ImageNet normalisation)
- Val sweep over confidence thresholds every epoch

In [ ]:

from fomo import FOMO

EPOCHS     = 10
BATCH      = 16
MODEL_SIZE = "m"   # "s" | "m" | "l"
PROJECT    = "runs/fomo"
RUN_NAME   = f"sjsu_headcount_{MODEL_SIZE}"

model = FOMO(model_path=None, size=MODEL_SIZE, nb_classes=1, device=device)
print(f"Model: FOMO-{MODEL_SIZE}")
print(f"  Input size : {model.input_size}×{model.input_size}")
print(f"  Grid size  : {model.input_size//8}×{model.input_size//8}")
print(f"  Classes    : {model.nb_classes}")

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# cosine decay by default
results = model.train(
    allow_experimental=True,
    data=str(data_yaml_path),
    epochs=EPOCHS,
    batch=BATCH,
    lr0=3e-4,
    eval_interval=1,
    workers=2,
    device=device,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    patience=0,
)

save_dir = Path(results["save_dir"])
print(f"\nTraining complete. Saved to: {save_dir}")


In [ ]:

from fomo import FOMO

weights_dir = save_dir / "weights"
best_pt  = weights_dir / "best.pt"
last_pt  = weights_dir / "last.pt"
ckpt_path = best_pt if best_pt.exists() else last_pt

trained = FOMO(str(ckpt_path), device=device)
print(f"Loaded: {ckpt_path.name}")
print(f"  family : {trained.family}")
print(f"  size   : {trained.size}")
print(f"  imgsz  : {trained.input_size}")
print(f"  nc     : {trained.nb_classes}")


## 6 — Export to TFLite (FP32)

We can now export the trained PyTorch model to a TFLite FP32 flatbuffer using the `export` API.

In [ ]:

fp32_path = trained.export(output_path=str(weights_dir / f"{RUN_NAME}_fp32.tflite"))
print(f"FP32 TFLite model exported to: {fp32_path}")


## 7 — INT8 Quantization

To deploy the model on edge devices (like the Arm Ethos-U NPU), we quantize the weights and activations to INT8.
This uses `INT8Quantizer` with a representative calibration dataset from the validation set.

In [ ]:

import numpy as np
from PIL import Image
from torchvision import transforms
from fomo.export import INT8Quantizer, INT8QuantizeConfig

val_img_dir = DATASET_ROOT / "valid" / "images"
calibration_images = sorted(val_img_dir.rglob("*.jpg"))

image_transform = transforms.Compose([
    transforms.Resize((trained.input_size, trained.input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

def get_calibration_data(image_paths, num_samples=150):
    """Yields representative data formatted for the LiteRT graph."""
    count = 0
    for img_path in image_paths:
        if count >= num_samples:
            return
        img = Image.open(img_path).convert("RGB")
        tensor = image_transform(img).unsqueeze(0).numpy()
        yield {"args_0": tensor}
        count += 1

num_samples = min(len(calibration_images), 150)
print(f"Using {num_samples} validation images for calibration.")
calib_iter = get_calibration_data(calibration_images, num_samples=num_samples)

config = INT8QuantizeConfig(num_calibration_samples=num_samples)
quantizer = INT8Quantizer()

int8_path = quantizer.quantize(
    fp32_tflite=fp32_path,
    calibration_data=calib_iter,
    config=config,
    output_path=str(weights_dir / f"{RUN_NAME}_int8.tflite")
)
print(f"INT8 quantized model exported to: {int8_path}")


## Vela Compilation

In [ ]:
!pip install -q ethos-u-vela

In [ ]:
!wget https://raw.githubusercontent.com/bencejdanko/Overhead-People-Counting-YOLOXNano-FOMO-Ethos-U55-NPU/refs/heads/main/configs/default_vela.ini

!vela /content/runs/fomo/sjsu_headcount_m/weights/sjsu_headcount_m_int8.tflite \
  --accelerator-config ethos-u55-256 \
  --config /content/default_vela.ini \
  --memory-mode Shared_Sram \
  --system-config Ethos_U55_High_End_Embedded \
  --output-dir vela_output \
  --optimise Performance